# Step 1 Solution Guide: Industry, Companies, and Filing Acquisition

Step 1 creates the foundation for the whole product. You choose one industry, list companies in that industry, download their 10-K filings, and save metadata that later notebooks can trust.

In this guide, the example industry is software publishers. Replace the example NAICS code, company names, and tickers with your team's actual industry.


## 1. Mount Drive and Create Paths

`BASE_DIR` is the project home folder. Every later path is built from it. The `.mkdir()` calls create folders if they do not already exist.


In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Not running in Colab. Use a local BASE_DIR path if needed.")

from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/project_sec10k_rag")

RAW_HTML_DIR = BASE_DIR / "data" / "raw_html"
EXTRACTED_ITEMS_DIR = BASE_DIR / "data" / "extracted_items"
OUTPUTS_DIR = BASE_DIR / "data" / "outputs"
CONFIG_DIR = BASE_DIR / "config"

for folder in [RAW_HTML_DIR, EXTRACTED_ITEMS_DIR, OUTPUTS_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Ready:", BASE_DIR)


Not running in Colab. Use a local BASE_DIR path if needed.
Ready: \content\drive\MyDrive\project_sec10k_rag


## 2. Document the Industry Choice

The NAICS code keeps the project focused. Do not mix unrelated industries. A good rationale explains why the industry is useful for financial analysis.


In [2]:
industry = {
    "naics_code": "5112",
    "industry_label": "Software Publishers",
    "rationale": (
        "Software publishers are useful for financial RAG because filings often discuss "
        "subscription revenue, cloud infrastructure costs, cybersecurity risk, and competition."
    ),
}

industry_df = pd.DataFrame([industry])
industry_df.to_csv(CONFIG_DIR / "industry_selection.csv", index=False)
industry_df


,naics_code,industry_label,rationale
0,5112,Software Publishers,Software publishers are useful for financial R...


## 3. Create a Company Scope Table

Use 5-10 public companies. The `ticker` is used for Yahoo Finance later. The `cik` is used for SEC filing lookup. If a CIK has leading zeros, store it as text.


In [3]:
companies = pd.DataFrame(
    [
        {"ticker": "MSFT", "company_name": "Microsoft Corporation", "cik": "0000789019", "naics_code": "5112"},
        {"ticker": "ADBE", "company_name": "Adobe Inc.", "cik": "0000796343", "naics_code": "5112"},
        {"ticker": "ORCL", "company_name": "Oracle Corporation", "cik": "0001341439", "naics_code": "5112"},
        {"ticker": "CRM", "company_name": "Salesforce, Inc.", "cik": "0001108524", "naics_code": "5112"},
        {"ticker": "INTU", "company_name": "Intuit Inc.", "cik": "0000896878", "naics_code": "5112"},
    ]
)

companies.to_csv(CONFIG_DIR / "companies.csv", index=False)
companies


,ticker,company_name,cik,naics_code
0,MSFT,Microsoft Corporation,0000789019,5112
1,ADBE,Adobe Inc.,0000796343,5112
2,ORCL,Oracle Corporation,0001341439,5112
3,CRM,"Salesforce, Inc.",0001108524,5112
4,INTU,Intuit Inc.,0000896878,5112


## 4. Build Standard SEC Filenames

Standard filenames make later cleaning and metadata joins much easier. The function below produces names that include the filing date, form type, CIK, accession number, and ticker.


In [4]:
import re

def clean_for_filename(value):
    value = str(value).upper()
    value = re.sub(r"[^A-Z0-9]+", "-", value)
    return value.strip("-")

def filing_filename(filing_date, form_type, cik, accession_number, company_name, ticker):
    safe_company = clean_for_filename(company_name)
    safe_accession = str(accession_number).replace("-", "")
    date_text = str(filing_date).replace("-", "")
    return f"{date_text}_{form_type}_{cik}_{safe_accession}_{safe_company}-({ticker}).html"

example_name = filing_filename(
    filing_date="2024-07-30",
    form_type="10-K",
    cik="0000789019",
    accession_number="0000950170-24-087843",
    company_name="Microsoft Corporation",
    ticker="MSFT",
)

example_name


'20240730_10-K_0000789019_000095017024087843_MICROSOFT-CORPORATION-(MSFT).html'

## 5. Create a Metadata Schema

Before downloading files, define the columns you need. This prevents teams from saving incomplete metadata and discovering the problem weeks later.


In [5]:
metadata_columns = [
    "accession_number", "form_type", "filing_date", "acceptance_datetime",
    "period_of_report", "company_name", "ticker", "cik", "sic_code",
    "sic_description", "organization_name", "fiscal_year_end",
    "state_of_incorporation", "irs_number", "sec_file_number", "film_number",
    "business_city", "business_state", "business_zip", "document_type",
    "document_period_end_date", "entity_file_number", "filer_category",
    "public_float", "shares_outstanding", "source_file", "generated_filename",
    "naics_code",
]

filing_metadata = pd.DataFrame(columns=metadata_columns)
filing_metadata.to_csv(OUTPUTS_DIR / "filing_metadata.csv", index=False)

print("Empty metadata file created with", len(metadata_columns), "columns")
filing_metadata.head()


Empty metadata file created with 28 columns


,accession_number,form_type,filing_date,acceptance_datetime,period_of_report,company_name,ticker,cik,sic_code,sic_description,...,business_zip,document_type,document_period_end_date,entity_file_number,filer_category,public_float,shares_outstanding,source_file,generated_filename,naics_code


## 6. Download 10-Ks for Every Company in the Vertical

This is the most important correction in Step 1. The downloader should not stop after one or two sample firms. It should loop over every row in `companies.csv`, request that company's SEC submission index, filter to 10-K filings from 2020 through the latest available filing year, and record a status for every expected company-year.

The code below uses the SEC company submissions endpoint. It downloads the filing's primary document into `data/raw_html/` and writes two audit files:

- `data/outputs/filing_metadata.csv`: one row per downloaded 10-K filing
- `data/outputs/download_manifest.csv`: one row per company-year attempt, including missing and error statuses

Why this fixes weak downstream results: Steps 2-5 can only clean, chunk, retrieve, and analyze filings that actually exist in the project folder. A manifest makes missing data visible instead of letting the pipeline quietly continue with a thin corpus.


In [6]:
import time
import requests
from datetime import datetime

START_FILING_YEAR = 2020
END_FILING_YEAR = datetime.today().year

# SEC requests must identify the requester. Replace this with your real BU email.
SEC_HEADERS = {
    "User-Agent": "AD698 student project your_email@example.com",
    "Accept-Encoding": "gzip, deflate",
}

def normalize_cik(cik):
    """Return a 10-digit CIK string, which is what the SEC submissions API expects."""
    return str(cik).strip().replace(".0", "").zfill(10)

def sec_submissions_url(cik):
    return f"https://data.sec.gov/submissions/CIK{normalize_cik(cik)}.json"

def get_company_submissions(cik):
    """Download the SEC submissions JSON for one company."""
    response = requests.get(sec_submissions_url(cik), headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()

def recent_filings_dataframe(submissions_json):
    """Convert the SEC 'recent filings' dictionary into a table."""
    recent = submissions_json["filings"]["recent"]
    return pd.DataFrame(recent)

def select_10k_filings(filings_df, start_year=START_FILING_YEAR, end_year=END_FILING_YEAR):
    """Keep annual 10-K filings whose filing date is in the desired year range."""
    filings = filings_df.copy()
    filings["filing_year"] = pd.to_datetime(filings["filingDate"], errors="coerce").dt.year
    is_10k = filings["form"].eq("10-K")
    in_range = filings["filing_year"].between(start_year, end_year)
    return filings.loc[is_10k & in_range].sort_values("filingDate")

def sec_document_url(cik, accession_number, primary_document):
    cik_int = str(int(str(cik).replace(".0", "")))
    accession_no_dashes = str(accession_number).replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_no_dashes}/{primary_document}"

def download_primary_document(cik, accession_number, primary_document, output_path):
    url = sec_document_url(cik, accession_number, primary_document)
    response = requests.get(url, headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    output_path.write_text(response.text, encoding="utf-8", errors="ignore")
    return url

download_rows = []
manifest_rows = []

for _, company in companies.iterrows():
    ticker = company["ticker"]
    company_name = company["company_name"]
    cik = company["cik"]
    naics_code = company["naics_code"]

    print(f"Checking {ticker}: {company_name}")

    try:
        submissions = get_company_submissions(cik)
        filings = select_10k_filings(recent_filings_dataframe(submissions))

        if filings.empty:
            manifest_rows.append(
                {
                    "ticker": ticker,
                    "company_name": company_name,
                    "cik": cik,
                    "naics_code": naics_code,
                    "filing_year": "",
                    "download_status": "no_10k_found_2020_to_latest",
                    "message": "No 10-K rows returned by SEC recent filings endpoint.",
                }
            )
            continue

        for _, filing in filings.iterrows():
            accession_number = filing["accessionNumber"]
            filing_date = filing["filingDate"]
            filing_year = int(filing["filing_year"])
            primary_document = filing["primaryDocument"]

            generated_filename = filing_filename(
                filing_date=filing_date,
                form_type="10-K",
                cik=normalize_cik(cik),
                accession_number=accession_number,
                company_name=company_name,
                ticker=ticker,
            )
            output_path = RAW_HTML_DIR / generated_filename

            try:
                document_url = download_primary_document(cik, accession_number, primary_document, output_path)
                status = "downloaded"
                message = ""
            except Exception as exc:
                document_url = sec_document_url(cik, accession_number, primary_document)
                status = "download_error"
                message = str(exc)

            row = {
                "accession_number": accession_number,
                "form_type": "10-K",
                "filing_date": filing_date,
                "acceptance_datetime": filing.get("acceptanceDateTime", ""),
                "period_of_report": filing.get("reportDate", ""),
                "company_name": company_name,
                "ticker": ticker,
                "cik": normalize_cik(cik),
                "sic_code": submissions.get("sic", ""),
                "sic_description": submissions.get("sicDescription", ""),
                "organization_name": "",
                "fiscal_year_end": submissions.get("fiscalYearEnd", ""),
                "state_of_incorporation": submissions.get("stateOfIncorporation", ""),
                "irs_number": submissions.get("ein", ""),
                "sec_file_number": filing.get("fileNumber", ""),
                "film_number": filing.get("filmNumber", ""),
                "business_city": submissions.get("addresses", {}).get("business", {}).get("city", ""),
                "business_state": submissions.get("addresses", {}).get("business", {}).get("stateOrCountry", ""),
                "business_zip": submissions.get("addresses", {}).get("business", {}).get("zipCode", ""),
                "document_type": primary_document,
                "document_period_end_date": filing.get("reportDate", ""),
                "entity_file_number": filing.get("fileNumber", ""),
                "filer_category": submissions.get("filerCategory", ""),
                "public_float": "",
                "shares_outstanding": "",
                "source_file": generated_filename,
                "generated_filename": generated_filename,
                "naics_code": naics_code,
                "filing_year": filing_year,
                "document_url": document_url,
                "local_path": str(output_path.relative_to(BASE_DIR)),
                "download_status": status,
                "message": message,
            }

            download_rows.append(row)
            manifest_rows.append(
                {
                    "ticker": ticker,
                    "company_name": company_name,
                    "cik": normalize_cik(cik),
                    "naics_code": naics_code,
                    "filing_year": filing_year,
                    "filing_date": filing_date,
                    "accession_number": accession_number,
                    "download_status": status,
                    "local_path": str(output_path.relative_to(BASE_DIR)),
                    "message": message,
                }
            )

            # Be polite to the SEC servers. The SEC limit is not a target.
            time.sleep(0.20)

    except Exception as exc:
        manifest_rows.append(
            {
                "ticker": ticker,
                "company_name": company_name,
                "cik": normalize_cik(cik),
                "naics_code": naics_code,
                "filing_year": "",
                "download_status": "company_level_error",
                "message": str(exc),
            }
        )

filing_metadata = pd.DataFrame(download_rows)
download_manifest = pd.DataFrame(manifest_rows)

# Keep the original course artifact name as a company-year coverage table.
industry_filing_panel = download_manifest.copy()
if not industry_filing_panel.empty:
    industry_filing_panel = industry_filing_panel.sort_values(
        ["ticker", "filing_year", "filing_date"],
        na_position="last",
    )

filing_metadata.to_csv(OUTPUTS_DIR / "filing_metadata.csv", index=False)
download_manifest.to_csv(OUTPUTS_DIR / "download_manifest.csv", index=False)
industry_filing_panel.to_csv(OUTPUTS_DIR / "industry_filing_panel.csv", index=False)

display(download_manifest.head())
display(download_manifest["download_status"].value_counts(dropna=False).to_frame("count"))


Checking MSFT: Microsoft Corporation
Checking ADBE: Adobe Inc.
Checking ORCL: Oracle Corporation
Checking CRM: Salesforce, Inc.
Checking INTU: Intuit Inc.


,ticker,company_name,cik,naics_code,filing_year,filing_date,accession_number,download_status,local_path,message
0,MSFT,Microsoft Corporation,0000789019,5112,2020,2020-07-30,0001564590-20-034944,downloaded,data\raw_html\20200730_10-K_0000789019_0001564...,
1,MSFT,Microsoft Corporation,0000789019,5112,2021,2021-07-29,0001564590-21-039151,downloaded,data\raw_html\20210729_10-K_0000789019_0001564...,
2,MSFT,Microsoft Corporation,0000789019,5112,2022,2022-07-28,0001564590-22-026876,downloaded,data\raw_html\20220728_10-K_0000789019_0001564...,
3,MSFT,Microsoft Corporation,0000789019,5112,2023,2023-07-27,0000950170-23-035122,downloaded,data\raw_html\20230727_10-K_0000789019_0000950...,
4,MSFT,Microsoft Corporation,0000789019,5112,2024,2024-07-30,0000950170-24-087843,downloaded,data\raw_html\20240730_10-K_0000789019_0000950...,


,count
download_status,
downloaded,28


## 7. Extract Core 10-K Items Into Separate Files

Step 2 expects item-level files, not whole 10-K documents. SEC filings are not perfectly standardized, so this starter extractor is intentionally conservative. It converts each downloaded HTML file to text, searches for major item headings, and saves selected sections into `data/extracted_items/`.

Use this as a transparent baseline. If a company has unusual filing formatting, inspect the raw HTML and improve the extraction rules for that company. The output table `extracted_item_metadata.csv` tells you which items were found.


In [7]:
# Run this install line in Colab if BeautifulSoup is missing:
# !pip -q install beautifulsoup4

import re
from bs4 import BeautifulSoup

ITEMS_TO_EXTRACT = ["1", "1A", "7", "7A", "8"]

def html_to_plain_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text("\n")
    text = re.sub(r"\n\s*\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text

def find_item_positions(text):
    # Matches headings such as "Item 1A.", "ITEM 7 -", and "Item 7A".
    pattern = re.compile(r"(?im)^\s*item\s+(1A|7A|1|7|8)\s*[\.\-: ]")
    positions = []
    for match in pattern.finditer(text):
        item_number = match.group(1).upper()
        positions.append((match.start(), item_number))

    # Table-of-contents entries often appear near the start. Keeping the later
    # repeated heading usually gets closer to the real section body.
    deduped = []
    seen = set()
    for start, item_number in sorted(positions, reverse=True):
        if item_number not in seen:
            deduped.append((start, item_number))
            seen.add(item_number)
    return sorted(deduped)

def extract_items_from_text(text, items_to_extract=ITEMS_TO_EXTRACT):
    positions = find_item_positions(text)
    extracted = {}

    for idx, (start, item_number) in enumerate(positions):
        end = positions[idx + 1][0] if idx + 1 < len(positions) else len(text)
        if item_number in items_to_extract:
            section_text = text[start:end].strip()
            if len(section_text.split()) >= 100:
                extracted[item_number] = section_text

    return extracted

if "filing_metadata" not in globals() or filing_metadata.empty:
    filing_metadata = pd.read_csv(OUTPUTS_DIR / "filing_metadata.csv")

extracted_rows = []

for _, filing in filing_metadata.query("download_status == 'downloaded'").iterrows():
    raw_path = BASE_DIR / filing["local_path"]
    if not raw_path.exists():
        continue

    html = raw_path.read_text(encoding="utf-8", errors="ignore")
    text = html_to_plain_text(html)
    extracted = extract_items_from_text(text)

    for item_number, item_text in extracted.items():
        item_filename = (
            f"{filing['filing_year']}_{filing['ticker']}_"
            f"{filing['accession_number'].replace('-', '')}_item_{item_number}.txt"
        )
        item_path = EXTRACTED_ITEMS_DIR / item_filename
        item_path.write_text(item_text, encoding="utf-8")

        extracted_rows.append(
            {
                "ticker": filing["ticker"],
                "company_name": filing["company_name"],
                "cik": filing["cik"],
                "naics_code": filing["naics_code"],
                "filing_year": filing["filing_year"],
                "filing_date": filing["filing_date"],
                "accession_number": filing["accession_number"],
                "item_number": item_number,
                "source_file": filing["source_file"],
                "extracted_file": item_filename,
                "word_count": len(item_text.split()),
            }
        )

extracted_item_metadata = pd.DataFrame(extracted_rows)
extracted_item_metadata.to_csv(OUTPUTS_DIR / "extracted_item_metadata.csv", index=False)

display(extracted_item_metadata.head())
display(extracted_item_metadata.groupby(["ticker", "item_number"]).size().unstack(fill_value=0) if not extracted_item_metadata.empty else extracted_item_metadata)


,ticker,company_name,cik,naics_code,filing_year,filing_date,accession_number,item_number,source_file,extracted_file,word_count
0,MSFT,Microsoft Corporation,0000789019,5112,2020,2020-07-30,0001564590-20-034944,1,20200730_10-K_0000789019_000156459020034944_MI...,2020_MSFT_000156459020034944_item_1.txt,8727
1,MSFT,Microsoft Corporation,0000789019,5112,2020,2020-07-30,0001564590-20-034944,1A,20200730_10-K_0000789019_000156459020034944_MI...,2020_MSFT_000156459020034944_item_1A.txt,10692
2,MSFT,Microsoft Corporation,0000789019,5112,2020,2020-07-30,0001564590-20-034944,7,20200730_10-K_0000789019_000156459020034944_MI...,2020_MSFT_000156459020034944_item_7.txt,9946
3,MSFT,Microsoft Corporation,0000789019,5112,2020,2020-07-30,0001564590-20-034944,7A,20200730_10-K_0000789019_000156459020034944_MI...,2020_MSFT_000156459020034944_item_7A.txt,275
4,MSFT,Microsoft Corporation,0000789019,5112,2020,2020-07-30,0001564590-20-034944,8,20200730_10-K_0000789019_000156459020034944_MI...,2020_MSFT_000156459020034944_item_8.txt,20504


item_number,1,1A,7,7A,8
ticker,,,,,
ADBE,7,7,7,7,7
CRM,3,3,3,3,3
INTU,6,6,6,6,6
MSFT,6,6,6,6,6
ORCL,6,6,6,6,6


## 8. Download Completeness Checks

This check asks the practical question: did every company in the vertical produce usable filings from 2020 through the latest available filing? If the answer is no, fix Step 1 before moving forward.


In [8]:
required_step1 = [
    CONFIG_DIR / "industry_selection.csv",
    CONFIG_DIR / "companies.csv",
    OUTPUTS_DIR / "filing_metadata.csv",
    OUTPUTS_DIR / "download_manifest.csv",
    OUTPUTS_DIR / "extracted_item_metadata.csv",
]

status = pd.DataFrame(
    {
        "path": [str(path.relative_to(BASE_DIR)) for path in required_step1],
        "exists": [path.exists() for path in required_step1],
    }
)

display(status)

if (OUTPUTS_DIR / "download_manifest.csv").exists():
    manifest = pd.read_csv(OUTPUTS_DIR / "download_manifest.csv")
    coverage = (
        manifest.assign(is_downloaded=manifest["download_status"].eq("downloaded"))
        .groupby(["ticker", "company_name"], dropna=False)
        .agg(
            filings_downloaded=("is_downloaded", "sum"),
            first_year=("filing_year", "min"),
            latest_year=("filing_year", "max"),
            statuses=("download_status", lambda s: ", ".join(sorted(set(map(str, s)))))
        )
        .reset_index()
    )
    display(coverage)

if (OUTPUTS_DIR / "extracted_item_metadata.csv").exists():
    extracted_meta = pd.read_csv(OUTPUTS_DIR / "extracted_item_metadata.csv")
    if not extracted_meta.empty:
        display(extracted_meta.groupby(["ticker", "item_number"]).size().unstack(fill_value=0))


,path,exists
0,config\industry_selection.csv,True
1,config\companies.csv,True
2,data\outputs\filing_metadata.csv,True
3,data\outputs\download_manifest.csv,True
4,data\outputs\extracted_item_metadata.csv,True


,ticker,company_name,filings_downloaded,first_year,latest_year,statuses
0,ADBE,Adobe Inc.,7,2020,2026,downloaded
1,CRM,"Salesforce, Inc.",3,2024,2026,downloaded
2,INTU,Intuit Inc.,6,2020,2025,downloaded
3,MSFT,Microsoft Corporation,6,2020,2025,downloaded
4,ORCL,Oracle Corporation,6,2020,2025,downloaded


item_number,1,1A,7,7A,8
ticker,,,,,
ADBE,7,7,7,7,7
CRM,3,3,3,3,3
INTU,6,6,6,6,6
MSFT,6,6,6,6,6
ORCL,6,6,6,6,6


## What to Submit for Step 1

Your notebook should show the industry choice, full company list, 2020-through-latest 10-K download loop, item extraction logic, and completeness checks. The key files are `filing_metadata.csv`, `download_manifest.csv`, `extracted_item_metadata.csv`, `industry_filing_panel.csv`, and extracted item files in `data/extracted_items/`.
